
# Appendix Notebook A1: Classical Search and QAOA for U.S. Equity Portfolio Selection

This notebook accompanies the optimisation case study in Module 1 of the integrated quantum finance review.
It replaces the earlier third-party Q4FuturePOP illustration with a fully reproducible local experiment built on the Nasdaq 100 daily price files already stored in this workspace.

The notebook addresses one concrete question: can a small, cardinality-constrained equity allocation problem be expressed as a QUBO and solved consistently by both a classical exact benchmark and a QAOA-based quantum workflow?



## Workflow Overview

1. Load six U.S. equities from the local `paper/data_nasdaq100_2024_2025/` directory.
2. Estimate annualised returns and the annualised covariance matrix from daily returns.
3. Form a binary mean-variance portfolio problem with an exact cardinality constraint of three assets.
4. Solve the problem classically by exhaustive enumeration.
5. Solve the equivalent QUBO with QAOA using Qiskit primitives.
6. Save a publication-ready figure for the review paper appendix and Module 1 discussion.

The experiment is intentionally small enough to remain transparent. The purpose is not to claim current quantum advantage, but to show modelling consistency, reproducibility, and a finance-relevant comparison on real local market data.


In [ ]:

from __future__ import annotations

from itertools import combinations
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from qiskit.primitives import StatevectorSampler
from qiskit_algorithms.minimum_eigensolvers import QAOA
from qiskit_algorithms.optimizers import COBYLA
from qiskit_optimization import QuadraticProgram
from qiskit_optimization.algorithms import MinimumEigenOptimizer
from qiskit_optimization.converters import QuadraticProgramToQubo


def locate_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'paper' / 'data_nasdaq100_2024_2025').exists():
            return candidate
    raise FileNotFoundError('Could not locate the repository root from the current working directory.')


ROOT = locate_repo_root(Path.cwd())
DATA_DIR = ROOT / 'paper' / 'data_nasdaq100_2024_2025'
FIG_DIR = ROOT / 'review_paper' / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

TICKERS = ['AAPL', 'AMD', 'AMGN', 'COST', 'CSX', 'GOOGL']
CARDINALITY = 3
RISK_AVERSION = 8.0
PENALTY = 15.0
QAOA_REPS = 2
SEED = 7


def load_close_series(ticker: str) -> pd.Series:
    csv_path = next(DATA_DIR.glob(f'{ticker}_*_daily.csv'))
    raw = pd.read_csv(csv_path, header=[0, 1])
    dates = pd.to_datetime(raw.iloc[1:, 0])
    closes = pd.to_numeric(raw.iloc[1:, 1], errors='coerce')
    return pd.Series(closes.to_numpy(), index=dates, name=ticker)



## Data Load and Return Estimation

We work with a six-asset universe to keep the QAOA instance interpretable and simulation-friendly.
The decision variable for each stock is binary: the stock is either included in the selected portfolio or excluded.
All selected positions are interpreted as equally weighted at the portfolio-construction stage.


In [ ]:

prices = pd.concat([load_close_series(ticker) for ticker in TICKERS], axis=1).sort_index().dropna()
returns = prices.pct_change().dropna()

mu = returns.mean() * 252
cov = returns.cov() * 252

display(prices.head())
display(mu.rename('annualised_return').to_frame())
display(pd.Series(np.diag(cov), index=TICKERS, name='annualised_variance').to_frame())

print(f'Sample period: {prices.index.min().date()} to {prices.index.max().date()}')
print(f'Observations: {len(returns)} daily returns')



## Portfolio Objective

The binary portfolio vector is $x \in \{0,1\}^n$ with $\sum_i x_i = K$.
We minimise a penalised mean-variance objective of the form

$$
\min_x \; \lambda x^\top \Sigma x - \mu^\top x + A\left(\sum_i x_i - K\right)^2,
$$

where $\lambda$ is the risk-aversion coefficient and $A$ is the QUBO penalty enforcing the cardinality constraint.
The classical benchmark evaluates every feasible three-stock combination exactly, while the quantum workflow solves the QUBO representation with QAOA.


In [ ]:

def equal_weight_metrics(combo: tuple[str, ...]) -> tuple[float, float]:
    weights = np.full(len(combo), 1.0 / len(combo))
    sub_cov = cov.loc[list(combo), list(combo)].to_numpy()
    annualised_return = float(mu.loc[list(combo)].mean())
    annualised_vol = float(np.sqrt(weights @ sub_cov @ weights))
    return annualised_return, annualised_vol


rows = []
for combo in combinations(TICKERS, CARDINALITY):
    x = np.array([1.0 if ticker in combo else 0.0 for ticker in TICKERS])
    objective = float(RISK_AVERSION * (x @ cov.to_numpy() @ x) - (mu.to_numpy() @ x))
    annualised_return, annualised_vol = equal_weight_metrics(combo)
    rows.append(
        {
            'portfolio': ' / '.join(combo),
            'tickers': combo,
            'objective': objective,
            'annualised_return': annualised_return,
            'annualised_vol': annualised_vol,
        }
    )

frontier = pd.DataFrame(rows).sort_values('objective').reset_index(drop=True)
classical_best = frontier.iloc[0].copy()
display(frontier.head(10))
print('Classical exact optimum:')
print(classical_best)



## QAOA Formulation and Solution

Qiskit is used here as a gate-based research stack. The implementation below builds a `QuadraticProgram`, converts the constrained model into a QUBO, and solves it with QAOA using a statevector sampler.
This is still a quantum simulation running on classical hardware, but it preserves the optimisation logic required for a future hardware migration.


In [ ]:

qp = QuadraticProgram(name='portfolio_qaoa_demo')
for ticker in TICKERS:
    qp.binary_var(name=ticker)

linear = {ticker: -float(mu.loc[ticker]) for ticker in TICKERS}
quadratic = {}
for i, ticker_i in enumerate(TICKERS):
    for j, ticker_j in enumerate(TICKERS[i:], start=i):
        quadratic[(ticker_i, ticker_j)] = float(RISK_AVERSION * cov.iloc[i, j])

qp.minimize(linear=linear, quadratic=quadratic)
qp.linear_constraint(linear={ticker: 1 for ticker in TICKERS}, sense='==', rhs=CARDINALITY, name='cardinality')

qubo = QuadraticProgramToQubo(penalty=PENALTY).convert(qp)
sampler = StatevectorSampler(seed=SEED)
qaoa = QAOA(sampler=sampler, optimizer=COBYLA(maxiter=200), reps=QAOA_REPS)
solver = MinimumEigenOptimizer(qaoa)
qaoa_result = solver.solve(qubo)

qaoa_selection = [ticker for ticker, value in zip(TICKERS, qaoa_result.x) if round(value) == 1]
qaoa_objective = float(RISK_AVERSION * (np.array(qaoa_result.x) @ cov.to_numpy() @ np.array(qaoa_result.x)) - (mu.to_numpy() @ np.array(qaoa_result.x)))
qaoa_return, qaoa_vol = equal_weight_metrics(tuple(qaoa_selection))

qaoa_summary = pd.Series(
    {
        'portfolio': ' / '.join(qaoa_selection),
        'objective': qaoa_objective,
        'annualised_return': qaoa_return,
        'annualised_vol': qaoa_vol,
        'qubo_fval': float(qaoa_result.fval),
        'constraint_sum': float(np.sum(qaoa_result.x)),
    },
    name='qaoa',
)

display(qaoa_summary.to_frame())
print(qaoa_result)



## Visual Comparison and Paper Figure Export

The first panel shows every feasible three-stock portfolio in risk-return space, coloured by the optimisation objective.
The second panel ranks all feasible portfolios by objective value and highlights the one recovered by QAOA.
If QAOA matches the classical optimum on this scaled problem, the figure makes that alignment visually explicit.


In [ ]:

qaoa_portfolio_label = ' / '.join(qaoa_selection)
frontier['rank'] = np.arange(1, len(frontier) + 1)
frontier['is_qaoa'] = frontier['portfolio'] == qaoa_portfolio_label
frontier['is_classical_best'] = frontier['portfolio'] == classical_best['portfolio']

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

scatter = axes[0].scatter(
    frontier['annualised_vol'],
    frontier['annualised_return'],
    c=frontier['objective'],
    cmap='viridis_r',
    s=90,
    edgecolor='black',
    linewidth=0.4,
    alpha=0.85,
)
axes[0].scatter(
    [classical_best['annualised_vol']],
    [classical_best['annualised_return']],
    color='crimson',
    s=220,
    marker='*',
    label='Classical exact optimum',
    zorder=5,
)
axes[0].scatter(
    qaoa_vol,
    [qaoa_return],
    color='navy',
    s=120,
    marker='D',
    label='QAOA solution',
    zorder=6,
)
axes[0].set_xlabel('Annualised volatility')
axes[0].set_ylabel('Annualised return')
axes[0].set_title('Feasible portfolios in risk-return space')
axes[0].legend(frameon=False, loc='best')
cbar = fig.colorbar(scatter, ax=axes[0])
cbar.set_label('Objective value (lower is better)')

axes[1].bar(frontier['rank'], frontier['objective'], color=['#b0b0b0' if not flag else '#1f77b4' for flag in frontier['is_qaoa']])
best_rank = int(frontier.loc[frontier['is_classical_best'], 'rank'].iloc[0])
qaoa_rank = int(frontier.loc[frontier['is_qaoa'], 'rank'].iloc[0])
axes[1].axvline(best_rank, color='crimson', linestyle='--', linewidth=1.6, label='Classical optimum rank')
axes[1].axvline(qaoa_rank, color='navy', linestyle=':', linewidth=1.6, label='QAOA rank')
axes[1].set_xlabel('Portfolio rank')
axes[1].set_ylabel('Objective value')
axes[1].set_title('Objective ranking across all feasible portfolios')
axes[1].legend(frameon=False, loc='best')

fig.suptitle('Classical exact search versus QAOA on a six-asset U.S. equity universe', fontsize=13)
fig.tight_layout()

figure_path = FIG_DIR / 'module_01_classical_vs_qaoa_portfolio_case.pdf'
fig.savefig(figure_path, bbox_inches='tight')
plt.close(fig)

summary = pd.DataFrame([
    {
        'method': 'Classical exact search',
        'portfolio': classical_best['portfolio'],
        'objective': float(classical_best['objective']),
        'annualised_return': float(classical_best['annualised_return']),
        'annualised_vol': float(classical_best['annualised_vol']),
    },
    {
        'method': 'QAOA (statevector simulation)',
        'portfolio': qaoa_portfolio_label,
        'objective': qaoa_objective,
        'annualised_return': qaoa_return,
        'annualised_vol': qaoa_vol,
    },
])
display(summary)
print(f'Figure saved to: {figure_path}')



## Interpretation

In this local benchmark, QAOA recovers the same three-stock portfolio as the classical exact search, which is the expected validation target for a small transparent instance.
The result should be read as a modelling and workflow confirmation rather than a claim of immediate computational superiority.

The real value of the exercise is threefold:
- it demonstrates how financial return and risk estimates can be encoded in a QUBO without outsourcing the example to another team's framework;
- it gives the paper a reproducible U.S.-equity case study tied to local data files;
- it creates a clean bridge from classical portfolio engineering to near-term quantum optimisation workflows.
